In [ ]:
#| default_exp vision_data

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch
from fastai.data.all import *
from fastai.vision.data import *
from fastMONAI.vision_core import *
from fastMONAI.vision_plot import find_max_slice
from plum import dispatch

# Vision data
> Build fastai `DataLoaders` for medical images and show batches, results and top losses.

## Prediction to mask

In [ ]:
#| export
def pred_to_multiclass_mask(pred: torch.Tensor) -> torch.Tensor:
    """Apply Softmax on the predicted tensor to rescale the values in the range [0, 1]
    and sum to 1. Then apply argmax to get the indices of the maximum value of all 
    elements in the predicted Tensor.

    Args:
        pred: [C,W,H,D] activation tensor.

    Returns: 
        Predicted mask.
    """
    
    pred = pred.softmax(dim=0)

    return pred.argmax(dim=0, keepdim=True)

In [ ]:
#| export
def batch_pred_to_multiclass_mask(pred: torch.Tensor) -> (torch.Tensor, int):
    """Convert a batch of predicted activation tensors to masks.
    
    Args:
        pred: [B, C, W, H, D] batch of activations.

    Returns:
        Tuple of batch of predicted masks and number of classes.
    """
    
    n_classes = pred.shape[1]
    pred = [pred_to_multiclass_mask(p) for p in pred]

    return torch.stack(pred), n_classes

In [ ]:
#| export
def pred_to_binary_mask(pred: torch.Tensor) -> torch.Tensor:
    """Apply Sigmoid function that squishes activations into a range between 0 and 1.
    Then we classify all values greater than or equal to 0.5 to 1, 
    and the values below 0.5 to 0.

    Args:
        pred: [B, C, W, H, D] or [C, W, H, D] activation tensor

    Returns:
        Predicted binary mask(s).
    """
    
    pred = torch.sigmoid(pred)

    return torch.where(pred >= 0.5, 1, 0)

## `DataBlock` for vision

In [ ]:
#| export
class MedDataBlock(DataBlock):
    """Container to quickly build dataloaders."""
    def __init__(self, blocks: list = None, dl_type: TfmdDL = None, getters: list = None,
                 n_inp: int | None = None, item_tfms: list = None, batch_tfms: list = None,
                 apply_reorder: bool = False, target_spacing: (int, list) = None, **kwargs):

        super().__init__(blocks, dl_type, getters, n_inp, item_tfms,
                         batch_tfms, **kwargs)

        MedBase.item_preprocessing(target_spacing, apply_reorder)

## `TransformBlock` for segmentation

In [ ]:
#| export
def MedMaskBlock():
    """Create a TransformBlock for medical masks."""
    return TransformBlock(type_tfms=MedMask.create)

## `DataLoaders` for vision

In [ ]:
#| export
class MedImageDataLoaders(DataLoaders):
    """Higher-level `MedDataBlock` API."""
    
    @classmethod
    @delegates(DataLoaders.from_dblock)
    def from_df(cls, df, valid_pct=0.2, seed=None, fn_col=0, folder=None, suff='',
                label_col=1, label_delim=None, y_block=None, valid_col=None,
                item_tfms=None, batch_tfms=None, apply_reorder=False, target_spacing=None,
                path=None, **kwargs):
        """Create from DataFrame.

        File names in `fn_col` are used as given. If `path` or `folder` is set, they are
        read relative to `path/folder`, as in fastai's `ImageDataLoaders.from_df`. The
        prefix is applied to single file names only, so give full paths for multi-channel
        rows that list several files.
        """
                    
        pref = ''
        if path is not None or folder is not None:
            pref = f"{Path(path or '.') / (folder or '')}{os.path.sep}"

        if y_block is None:
            is_multi = (is_listy(label_col) and len(label_col) > 1) or label_delim is not None
            y_block = MultiCategoryBlock if is_multi else CategoryBlock

        splitter = (RandomSplitter(valid_pct, seed=seed) 
                    if valid_col is None else ColSplitter(valid_col))

        dblock = MedDataBlock(
            blocks=(ImageBlock(cls=MedImage), y_block),
            get_x=ColReader(fn_col, pref=pref, suff=suff),
            get_y=ColReader(label_col, label_delim=label_delim),
            splitter=splitter,
            item_tfms=item_tfms,
            batch_tfms=batch_tfms,
            apply_reorder=apply_reorder,
            target_spacing=target_spacing
        )

        dls = cls.from_dblock(dblock, df, path=path or '.', **kwargs)
        # Retain column identities; actual membership remains in train_ds/valid_ds.
        dls._img_col = fn_col
        dls._mask_col = label_col
        return dls

In [ ]:
#| hide
# batch_tfms reach the dataloaders, and file names are read relative to folder.
import tempfile
import torchio as tio
from fastcore.test import test_eq

class _MarkBatch(Transform):
    order = 99
    def encodes(self, x: MedImage): return x

with tempfile.TemporaryDirectory() as _tmp:
    _names = [f'img{_i}.nii.gz' for _i in range(5)]
    for _name in _names:
        tio.ScalarImage(tensor=torch.rand(1, 4, 4, 4), affine=np.eye(4)).save(Path(_tmp) / _name)
    _df = pd.DataFrame({'fn': _names, 'path': [str(Path(_tmp) / _n) for _n in _names],
                        'label': ['a', 'b', 'a', 'b', 'a']})
    _dls = MedImageDataLoaders.from_df(_df, fn_col='fn', folder=_tmp, label_col='label',
                                       batch_tfms=[_MarkBatch()], bs=2, num_workers=0)
    assert any(isinstance(_t, _MarkBatch) for _t in _dls.train.after_batch.fs)
    test_eq(tuple(_dls.one_batch()[0].shape), (2, 1, 4, 4, 4))
    _dls = MedImageDataLoaders.from_df(_df, fn_col='path', label_col='label', bs=2, num_workers=0)
    test_eq(tuple(_dls.one_batch()[0].shape), (2, 1, 4, 4, 4))


## Show methods

### Show batch of images

In [ ]:
#| export
@dispatch
def show_batch(x: MedImage, y, samples, ctxs=None, max_n=6, nrows=None, 
               ncols=None, figsize=None, channel=0, slice_index=None,
               anatomical_plane=0, **kwargs):
    '''Showing a batch of samples for classification and regression tasks.'''

    if ctxs is None: 
        ctxs = get_grid(min(len(samples), max_n), nrows=nrows, ncols=ncols, figsize=figsize)
        
    n = 1 if y is None else 2
    
    for i in range(n):
        ctxs = [
            b.show(ctx=c, channel=channel, slice_index=slice_index, anatomical_plane=anatomical_plane, **kwargs)
            for b,c,_ in zip(samples.itemgot(i),ctxs,range(max_n))
        ]

    plt.tight_layout()
    
    return ctxs

In [ ]:
#| export
@dispatch
def show_batch(x: MedImage, y: MedMask, samples, ctxs=None, max_n=6, nrows=None,
               ncols=None, figsize=None, channel=0, slice_index=None,
               anatomical_plane=0, **kwargs):
                   
    """Showing a batch of decoded segmentation samples. `kwargs` are passed to each `show`."""
                   
    nrows = min(len(samples), max_n)
    ncols = x.shape[1] + 1
    imgs = []
    slice_idxs = []

    fig, axs = subplots(nrows, ncols, figsize=figsize)
    axs = axs.flatten()

    for img, mask in zip(x, y):
        im_channels = [MedImage(c_img[None]) for c_img in img]
        im_channels.append(MedMask(mask))
        imgs.extend(im_channels)

        idx = find_max_slice(mask[0].numpy(), anatomical_plane) if slice_index is None else slice_index
        slice_idxs.extend([idx] * (img.shape[0] + 1))
                   
    ctxs = [im.show(ax=ax, slice_index=idx, anatomical_plane=anatomical_plane, **kwargs)
            for im, ax, idx in zip(imgs, axs, slice_idxs)]

    plt.tight_layout()

    return ctxs

In [ ]:
#| hide
# Extra keyword arguments reach each image's show().
import tempfile
import torchio as tio
from fastcore.test import test_eq
with tempfile.TemporaryDirectory() as _tmp:
    _rows = []
    for _i in range(2):
        _img, _msk = str(Path(_tmp) / f'img{_i}.nii.gz'), str(Path(_tmp) / f'mask{_i}.nii.gz')
        tio.ScalarImage(tensor=torch.rand(1, 8, 8, 8), affine=np.eye(4)).save(_img)
        _m = torch.zeros(1, 8, 8, 8)
        _m[0, 2:5, 2:5, 2:5] = 1
        tio.LabelMap(tensor=_m, affine=np.eye(4)).save(_msk)
        _rows.append({'img': _img, 'mask': _msk})
    _dblock = MedDataBlock(blocks=(ImageBlock(cls=MedImage), MedMaskBlock), get_x=ColReader('img'),
                           get_y=ColReader('mask'), splitter=RandomSplitter(valid_pct=0.5, seed=0))
    _dls = _dblock.dataloaders(pd.DataFrame(_rows), bs=1, num_workers=0)
    _dls.show_batch(max_n=1, anatomical_plane=0, voxel_size=[1, 1, 2], cmap='hot')
    _ax = plt.gcf().axes[0]
    test_eq(_ax.get_aspect(), 2.0)
    test_eq(_ax.images[0].get_cmap().name, 'hot')
    plt.close('all')

### Show results

In [ ]:
#| export
@dispatch
def show_results(x: MedImage, y: torch.Tensor, samples, outs, ctxs=None, max_n: int = 6,
                 nrows: int | None = None, ncols: int | None = None, figsize=None, channel: int = 0,
                 slice_index: int | None = None, anatomical_plane: int = 0, **kwargs):
    """Showing samples and their corresponding predictions for regression tasks."""

    if ctxs is None:
        ctxs = get_grid(min(len(samples), max_n), nrows=nrows,
                        ncols=ncols, figsize=figsize)

    for i in range(len(samples[0])):
        ctxs = [
            b.show(ctx=c, channel=channel, slice_index=slice_index,
                   anatomical_plane=anatomical_plane, **kwargs)
            for b, c, _ in zip(samples.itemgot(i), ctxs, range(max_n))
        ]

    for i in range(len(outs[0])):
        ctxs = [
            b.show(ctx=c, **kwargs)
            for b, c, _ in zip(outs.itemgot(i), ctxs, range(max_n))
        ]

    return ctxs

In [ ]:
#| export
@dispatch
def show_results(x: MedImage, y: TensorCategory, samples, outs, ctxs=None, 
                 max_n: int = 6, nrows: int | None = None, ncols: int | None = None, figsize=None, channel: int = 0, 
                 slice_index: int | None = None, anatomical_plane: int = 0, **kwargs):
    """Showing samples and their corresponding predictions for classification tasks."""

    if ctxs is None: 
        ctxs = get_grid(min(len(samples), max_n), nrows=nrows, 
                        ncols=ncols, figsize=figsize)
    
    for i in range(2):
        ctxs = [b.show(ctx=c, channel=channel, slice_index=slice_index, 
                       anatomical_plane=anatomical_plane, **kwargs) 
                for b, c, _ in zip(samples.itemgot(i), ctxs, range(max_n))]

    ctxs = [r.show(ctx=c, color='green' if b == r else 'red', **kwargs) 
            for b, r, c, _ in zip(samples.itemgot(1), outs.itemgot(0), ctxs, range(max_n))]

    return ctxs

In [ ]:
#| export
@dispatch
def show_results(x: MedImage, y: MedMask, samples, outs, ctxs=None, max_n: int = 6, 
                 nrows: int | None = None, ncols: int = 3, figsize=None, channel: int = 0, 
                 slice_index: int | None = None, anatomical_plane: int = 0, **kwargs):
    """Showing decoded samples and their corresponding predictions for segmentation tasks."""
    
    if ctxs is None: 
        total_slots = 3 * min(len(samples), max_n)
        ctxs = get_grid(total_slots, nrows=nrows, ncols=ncols, 
                        figsize=figsize, double=False, title='Image/Target/Prediction')
    
    slice_idxs = [find_max_slice(mask[0].numpy(), anatomical_plane) if slice_index is None else slice_index for mask in y]

    ctxs[::3] = [b.show(ctx=c, channel=channel, slice_index=idx, anatomical_plane=anatomical_plane, **kwargs)
                 for b, c, idx in zip(x, ctxs[::3], slice_idxs)]

    for i in range(2):
        current_channel = 0 if i == 1 else channel
        ctxs[1::3] = [b.show(ctx=c, channel=current_channel, slice_index=idx, 
                            anatomical_plane=anatomical_plane, **kwargs) 
                     for b, c, _, idx in zip(samples.itemgot(i), ctxs[1::3], range(2 * max_n), slice_idxs)]

    for index, o in enumerate([samples, outs]):
        current_channel = 0 if index == 1 else channel
        ctxs[2::3] = [b.show(ctx=c, channel=current_channel, slice_index=idx, 
                             anatomical_plane=anatomical_plane, **kwargs) 
                      for b, c, _, idx in zip(o.itemgot(0), ctxs[2::3], range(2 * max_n), slice_idxs)]

    return ctxs

### Show top losses

In [ ]:
#| export
@dispatch
def plot_top_losses(x: MedImage, y: torch.Tensor, samples, outs, raws, losses, nrows: int | None = None,
                    ncols: int | None = None, figsize=None, channel: int = 0, slice_index: int | None = None,
                    anatomical_plane: int = 0, **kwargs):
    """Show images in top_losses for regression, with their prediction, target and loss."""

    axs = get_grid(len(samples), nrows=nrows, ncols=ncols, figsize=figsize, title='Prediction/Actual/Loss')

    for ax, s, r, l in zip(axs, samples, raws, losses):
        s[0].show(ctx=ax, channel=channel, slice_index=slice_index, anatomical_plane=anatomical_plane, **kwargs)
        ax.set_title(f'{r.max().item():.2f}/{s[1]} / {l.item():.2f}')

In [ ]:
#| export
@dispatch
def plot_top_losses(x: MedImage, y: TensorCategory, samples, outs, raws, losses, nrows: int | None = None, 
                    ncols: int | None = None, figsize=None, channel: int = 0, slice_index: int | None = None, 
                    anatomical_plane: int = 0, **kwargs):
    """Show images in top_losses for classification, with their predicted class, actual class,
    loss and probability of the predicted class."""

    axs = get_grid(len(samples), nrows=nrows, ncols=ncols, figsize=figsize,
                   title='Prediction/Actual/Loss/Probability')

    for ax, s, o, r, l in zip(axs, samples, outs, raws, losses):
        s[0].show(ctx=ax, channel=channel, slice_index=slice_index, anatomical_plane=anatomical_plane, **kwargs)
        ax.set_title(f'{o[0]}/{s[1]} / {l.item():.2f} / {r.max().item():.2f}')

In [ ]:
#| export
@dispatch
def plot_top_losses(x: MedImage, y: TensorMultiCategory, samples, outs, raws, 
                    losses, nrows: int | None = None, ncols: int | None = None, figsize=None, 
                    channel: int = 0, slice_index: int | None = None, 
                    anatomical_plane: int = 0, **kwargs):
    """Show images in top_losses for multi-label classification, with a table of
    their targets, predictions, probabilities and losses."""
    axs = get_grid(len(samples), nrows=nrows, ncols=ncols, figsize=figsize)

    for i, (ax, s) in enumerate(zip(axs, samples)):
        s[0].show(ctx=ax, title=f'Image {i}', channel=channel, 
                  slice_index=slice_index, anatomical_plane=anatomical_plane, **kwargs)

    rows = get_empty_df(len(samples))
    outs = L(s[1:] + o + (TitledStr(r), TitledFloat(l.item())) 
             for s, o, r, l in zip(samples, outs, raws, losses))

    for i, l in enumerate(["target", "predicted", "probabilities", "loss"]):
        rows = [b.show(ctx=r, label=l, channel=channel, slice_index=slice_index, 
                       anatomical_plane=anatomical_plane, **kwargs) 
                for b, r in zip(outs.itemgot(i), rows)]

    display_df(pd.DataFrame(rows))

In [ ]:
#| export
@dispatch
def plot_top_losses(x: MedImage, y: MedMask, samples, outs, raws, losses, **kwargs):
    """Top losses are not plotted for segmentation."""
    raise NotImplementedError('plot_top_losses does not support segmentation; use show_results instead.')

In [ ]:
#| hide
# Regression titles show prediction/target/loss; classification titles start with the predicted class.
import tempfile
import torchio as tio
from torch import nn
from fastai.learner import Learner
from fastai.losses import CrossEntropyLossFlat, L1LossFlat
from fastai.interpret import Interpretation, ClassificationInterpretation
from fastcore.test import test_eq, test_fail

class _TinyNet(nn.Module):
    def __init__(self, n_out):
        super().__init__()
        self.conv, self.head = nn.Conv3d(1, 2, 3, padding=1), nn.Linear(2, n_out)
    def forward(self, x): return self.head(self.conv(x).mean((2, 3, 4)))

def _top_loss_titles(df, y_block, label_col, n_out, loss_func, interp_cls):
    dblock = MedDataBlock(blocks=(ImageBlock(cls=MedImage), y_block), get_x=ColReader('path'),
                          get_y=ColReader(label_col), splitter=ColSplitter('is_val'))
    learn = Learner(dblock.dataloaders(df, bs=2, num_workers=0), _TinyNet(n_out), loss_func=loss_func)
    interp_cls.from_learner(learn).plot_top_losses(k=2, anatomical_plane=2)
    titles = [ax.get_title() for ax in plt.gcf().axes]
    plt.close('all')
    return titles

with tempfile.TemporaryDirectory() as _tmp:
    _paths = [str(Path(_tmp) / f'img{_i}.nii.gz') for _i in range(6)]
    for _p in _paths:
        tio.ScalarImage(tensor=torch.rand(1, 8, 8, 8), affine=np.eye(4)).save(_p)
    _df = pd.DataFrame({'path': _paths, 'label': list('ababab'), 'age': [20., 21., 22., 23., 24., 25.],
                        'is_val': [False] * 4 + [True] * 2})
    _reg = _top_loss_titles(_df, RegressionBlock, 'age', 1, L1LossFlat(), Interpretation)
    _cls = _top_loss_titles(_df, CategoryBlock, 'label', 2, CrossEntropyLossFlat(), ClassificationInterpretation)

test_eq(len(_reg), 2)
assert all(len(_t.split('/')) == 3 and _t.split('/')[1].split(' ')[0] in ('24.0', '25.0') for _t in _reg), _reg
test_eq(len(_cls), 2)
assert all(len(_t.split('/')) == 4 and _t.split('/')[0] in ('a', 'b') for _t in _cls), _cls
_seg = MedMask(torch.zeros(1, 1, 2, 2, 2))
test_fail(lambda: plot_top_losses(MedImage(torch.zeros(1, 1, 2, 2, 2)), _seg, [], [], [], []),
          contains='segmentation')